In [2]:
import sounddevice as sd
import numpy as np
import soundfile as sf
import keyboard


In [3]:

def print_device_info():
    devices = sd.query_devices()
    for idx, device in enumerate(devices):
        name = device['name']
        inputs = device['max_input_channels']
        outputs = device['max_output_channels']
        
        print(f"ID: {idx} | Name: {name} | Inputs: {inputs} | Outputs: {outputs}")

print_device_info()

ID: 0 | Name: HDA NVidia: HDMI 1 (hw:0,7) | Inputs: 0 | Outputs: 8
ID: 1 | Name: HDA NVidia: HDMI 2 (hw:0,8) | Inputs: 0 | Outputs: 8
ID: 2 | Name: HDA NVidia: HDMI 3 (hw:0,9) | Inputs: 0 | Outputs: 8
ID: 3 | Name: sof-hda-dsp: - (hw:1,0) | Inputs: 2 | Outputs: 2
ID: 4 | Name: sof-hda-dsp: - (hw:1,3) | Inputs: 0 | Outputs: 2
ID: 5 | Name: sof-hda-dsp: - (hw:1,4) | Inputs: 0 | Outputs: 2
ID: 6 | Name: sof-hda-dsp: - (hw:1,5) | Inputs: 0 | Outputs: 2
ID: 7 | Name: sof-hda-dsp: - (hw:1,6) | Inputs: 2 | Outputs: 0
ID: 8 | Name: sof-hda-dsp: - (hw:1,7) | Inputs: 2 | Outputs: 0
ID: 9 | Name: sof-hda-dsp: - (hw:1,31) | Inputs: 0 | Outputs: 2


In [4]:
# Global state for the soundboard audio
sound_data = None
sound_index = 0

def play_sound(file_path):
    global sound_data, sound_index
    data, _ = sf.read(file_path, dtype='float32')
    sound_data = data
    sound_index = 0

def mixing_callback(indata, outdata, frames, time, status):
    global sound_data, sound_index
    
    mixed_audio = indata.copy()
    
    if sound_data is not None:
        remaining = len(sound_data) - sound_index
        chunk_size = min(frames, remaining)
        
        if chunk_size > 0:
            # Add the soundboard audio to the microphone audio
            mixed_audio[:chunk_size] += sound_data[sound_index:sound_index + chunk_size]
            sound_index += chunk_size
        else:
            sound_data = None 
            
    outdata[:] = mixed_audio

def start_audio_engine(input_id, output_id, sample_rate=44100):
    stream = sd.Stream(
        device=(input_id, output_id),
        samplerate=sample_rate,
        channels=2,
        callback=mixing_callback
    )
    return stream

In [6]:

def setup_hotkeys():
    keyboard.add_hotkey('ctrl+shift+a', lambda: play_sound("sounds/dry-fart.mp3"))
    keyboard.add_hotkey('f9', lambda: play_sound("sounds/assumptions.mp3"))

setup_hotkeys()

ImportError: You must be root to use this library on linux.